<a href="https://colab.research.google.com/github/GuilhermeFernandez/climate_resilient_pastures/blob/main/Esboco_Projeto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Parte 1
### Objetivos:

Foco: Definir a base estática e limpar os polígonos.

Devo gerar um GeoJSON/Dataframe contendo:

1.   ID_Area: Identificador único; ✅
2.   nm_mun: município do polígono; ✅
3. classe_manejo: Categoria fixa (Extensivo, Rotacionado, SSP Nativas, SSP Exóticas);
4. geometry: Polígono da propriedade; ✅
5. area_ha: Área calculada da geometria útil. ✅
6. date_implementation: Ano em que o sistema foi implantado.
7. %_cobertura_arborea: calculada com modelo baseado em CBERS-4A, imagens de 2025 (se possível), máscara de recorte com buffer de 1pixel 8x8.










In [ ]:
# @title Importando Bibliotecas necessárias
!pip install geobr
import sys, os, geobr
import geopandas as gpd
import pandas as pd

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.0/338.0 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 94.0 MB/s eta 0:00:00
  Attempting uninstall: shapely
    Found existing installation: shapely 2.1.2
    Uninstalling shapely-2.1.2:
      Successfully uninstalled shapely-2.1.2
  Attempting uninstall: lxml
    Found existing installation: lxml 6.0.2
    Uninstalling lxml-6.0.2:
      Successfully uninstalled lxml-6.0.2
  Attempting uninstall: geopandas
    Found existing installation: geopandas 1.1.2
    Uninstalling geopandas-1.1.2:
      Successfully uninstalled geopandas-1.1.2


In [ ]:
# @title Definindo diretório
path = '/content/drive/MyDrive/Colab Notebooks/Esboco_Projeto'
files = [f for f in os.listdir(path) if f.endswith('.shp')]

In [ ]:
# @title Extraindo geometrias dos shapefiles no diretório:
gdfs = []

for file in files:
    full_path = os.path.join(path, file)
    try:
        gdf_temp = gpd.read_file(full_path)
        gdfs.append(gdf_temp)
    except Exception as e:
        print(f"Erro ao ler {file}: {e}")

if gdfs:
    gdf = pd.concat(gdfs, ignore_index=True)

    #Ajuste de Projeção para Cálculo de Área:
    if gdf.crs is None:
        gdf.set_crs(epsg=32721, inplace=True) # Assumindo UTM 21S se não tiver nada
    else:
        gdf = gdf.to_crs(epsg=32721) # Convertendo para UTM 21S se estiver em outro

    #Calculo da área:
    gdf['Area_ha'] = gdf.area / 10000

    #Converte para o CRS final de saída (SIRGAS 2000 Lat/Lon):
    gdf = gdf.to_crs(epsg=4674)

    #Cria o ID para a Área de Estudo e Salva
    gdf['ID_Area'] = gdf.index + 1

    output_file = os.path.join(path, 'study_areas.geojson')
    gdf.to_file(output_file, driver='GeoJSON')
    print("Arquivo salvo com sucesso!")
else:
    print("Nenhum arquivo shapefile encontrado.")

Arquivo salvo com sucesso!


In [ ]:
# @title Extraindo município das coordenadas a partir de IBGE 2024
municipalities = geobr.read_municipality(code_muni='MT', year=2024)
gdf_mun = gpd.sjoin(gdf, municipalities[['name_muni', 'geometry']],
                             how='left',
                             predicate='intersects')
gdf_mun = gdf_mun.drop(['index_right', 'id'], axis=1)
gdf_mun.columns = ['geometry','area_ha','ID_Area','nm_mun']
gdf_mun.set_index('ID_Area')

,geometry,area_ha,nm_mun
ID_Area,,,
1,"POLYGON ((-55.38248 -10.21095, -55.38102 -10.2...",165.78976,Nova Guarita


## Parte 2
### Objetivos:

Foco: Processamento pesado. Classificar cobertura e extrair índices para cada ano/mês dos 40 anos. A área deve ser recortada pela máscara gerada na Parte 1 para retirada de cobertura arbórea. Deve iterar sobre a coleção Landsat e CHIRPS e gerar um Dataframe contendo:

1.   ID_Area: Identificador único; ✅
2.   date: Ano;
3. ndvi_value: Média (?) do NDVI apenas nos pixels de pasto;
4. ndmi_value: Média (?) do NDMI;
5. lst_value: Média (?) da temperatura de superfície;
6. precipitation_mm: Acumulado de chuva do CHIRPS para o período;
7. precip_anomalies: para cada ano, True ou False.


## Parte 3
### Objetivos:

Foco: Transformar dados em resultados.

Deve ler o Dataframe do Script 2 e gerar um Dataframe de Resultados contendo:


1.   ID_Area: Identificador único; ✅
2.   Index: Coluna indicando qual índice (NDVI, NDMI ou LST):
> 3. Senslope_trend_pre: Valor da inclinação da tendência ao longo da série;
> 4. Senslope_trend_post: Valor da inclinação da tendência ao longo da série após implementação, se houver;
> 5. trend_significance_pre: Valor-p do teste Mann-Kendall;
> 6. trend_significance_post: Valor-p do teste Mann-Kendall;
> 7. z_score_drought: O desvio padrão do índice durante os anos de anomalia climática (secas identificadas = precip_anomalies == True);
> 8. comparative_resilience: Rótulo final (Brightspot, Hotspot ou Neutro) baseado na comparação com vizinhos em períodos de anomalia climática (secas identificadas = precip_anomalies == True).



